# Calling Functions with Semantic Kernel

This notebook demonstrates how to call C# functions (plugins) from natural language prompts using the Microsoft Semantic Kernel in .NET. You'll learn to define, register, and invoke functions that the AI model can call automatically during a conversation.

**Step 1:** Install NuGet packages

Install the required NuGet packages for Semantic Kernel and DotNetEnv to enable AI-powered workflows and environment variable management.

In [ ]:
// Import Semantic Kernel
#r "nuget: Microsoft.SemanticKernel, 1.23.0"
#r "nuget: DotNetEnv, 3.1.0"

**Step 2:** Load environment variables

Load environment variables from a `.env` file if present. This helps manage secrets and configuration settings for your application.

In [ ]:
using DotNetEnv;
using System.IO;

var envFilePath = Path.Combine(Environment.CurrentDirectory, "../..", ".env");
if (File.Exists(envFilePath))
{
    Env.Load(envFilePath);
    Console.WriteLine($"Loaded environment variables from {envFilePath}");
}
else
{
    Console.WriteLine($"No .env file found at {envFilePath}");
}

**Step 3:** Instantiate the Semantic Kernel

Create and configure a Kernel instance, which will be used to interact with AI models.

In [ ]:
using System.ClientModel;
using OpenAI;
using Microsoft.SemanticKernel;
using Microsoft.SemanticKernel.ChatCompletion;
using System.Text;

OpenAIClient client = null;
if(Environment.GetEnvironmentVariable("USE_AZURE_OPENAI") == "true")
{
    // Configure Azure OpenAI client
    var azureEndpoint = Environment.GetEnvironmentVariable("AZURE_OPENAI_ENDPOINT");
    var apiKey = Environment.GetEnvironmentVariable("AZURE_OPENAI_API_KEY");
    client = new OpenAIClient(new ApiKeyCredential(apiKey), new OpenAIClientOptions { Endpoint = new Uri(azureEndpoint) });
}
else if(Environment.GetEnvironmentVariable("USE_OPENAI") == "true")
{
    // Configure OpenAI client
    var apiKey = Environment.GetEnvironmentVariable("OPENAI_API_KEY");
    client = new OpenAIClient(new ApiKeyCredential(apiKey));
}
else if(Environment.GetEnvironmentVariable("USE_GITHUB") == "true")
{
    // Configure GitHub model client
    var uri = Environment.GetEnvironmentVariable("GITHUB_MODEL_ENDPOINT");
    var apiKey = Environment.GetEnvironmentVariable("GITHUB_TOKEN");
    client = new OpenAIClient(new ApiKeyCredential(apiKey), new OpenAIClientOptions { Endpoint = new Uri(uri) });
}

var modelId = Environment.GetEnvironmentVariable("MODEL");
// Create a chat completion service
var builder = Kernel.CreateBuilder();
builder.AddOpenAIChatCompletion(modelId, client);

// Get the chat completion service
Kernel kernel = builder.Build();
var chat = kernel.GetRequiredService<IChatCompletionService>(); 

**Step 4:** Create a semantic function

Define a semantic function using a prompt template. This function will summarize input content.

In [ ]:
string skPrompt = """
{{$input}}

Summarize the content above.
""";

**Step 5:** Configure function calling settings

Set up the execution settings for function calling, such as enabling automatic function selection.

In [ ]:
using Microsoft.SemanticKernel;
using Microsoft.SemanticKernel.Connectors.OpenAI;

#pragma warning disable SKEXP0001
var executionSettings = new OpenAIPromptExecutionSettings 
{
    FunctionChoiceBehavior = FunctionChoiceBehavior.Auto()
};
#pragma warning restore SKEXP0001


**Step 6:** Define and register a C# plugin function

Define a C# function (plugin) that returns random weather information, and register it with the kernel.

In [ ]:
// Define a function that returns random weather
using System;
using Microsoft.SemanticKernel;

public class WeatherPlugin
{
    [KernelFunction]
    public string GetRandomWeather()
    {
        var weatherOptions = new[] { "Sunny", "Cloudy", "Rainy", "Windy", "Stormy", "Snowy" };
        var rnd = new Random();
        var temp = rnd.Next(10, 35);
        var weather = weatherOptions[rnd.Next(weatherOptions.Length)];
        return $"The weather is {weather} and {temp}°C.";
    }
}

**Step 7:** Call the plugin function using function calling

Send a prompt to the kernel that triggers the function call, and display the AI's response and function call details.

In [11]:
// Register the WeatherPlugin with the kernel
var plugin = new WeatherPlugin();
kernel.ImportPluginFromObject(plugin, "weather");

// Create a prompt that will trigger the function call
string weatherPrompt = "What is the weather today?";

// Use the execution settings for function calling
var result = await chat.GetChatMessageContentAsync(weatherPrompt, executionSettings, kernel);
Console.WriteLine($"AI: {result.Content}");

AI: The weather today is sunny with a temperature of 22°C.
